# **Performance metrics of trained models**

## **Give google drive access**


In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
%cd "/content/drive/My Drive/Colab Notebooks/Turbidity model - Hang_s thesis/ML_turbidity_prediction-main"

Mounted at /content/drive
/content/drive/My Drive/Colab Notebooks/Turbidity model - Hang_s thesis/ML_turbidity_prediction-main


## **Metrics scripts**


In [3]:
# -*- coding: utf-8 -*-
"""
Evaluation metrics for turbidity forecasting -- regression and event classification.

Designed to replace the per-model metric blocks in `predictor.py` with ONE set of
functions, so every model is scored identically and the numbers are comparable.

    from evaluation import (regression_metrics, persistence_forecast,
                            block_bootstrap_ci, diebold_mariano,
                            classification_metrics, threshold_sweep,
                            event_metrics)

    base = persistence_forecast(obs, horizon=1)
    m = regression_metrics(obs, pred, baseline=base, threshold=25.0)
    print(pd.Series(m))

Everything expects values in ORIGINAL UNITS (FTU), not scaled.
"""

from math import ceil

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             confusion_matrix, matthews_corrcoef,
                             mean_absolute_error, mean_squared_error,
                             precision_score, recall_score, roc_auc_score)


# =============================================================================
# REGRESSION
# =============================================================================
def persistence_forecast(observed, horizon=1):
    """The forecast every model must beat: 'turbidity in h hours = turbidity now'.

    Returns an array the same length as `observed`, with the first `horizon`
    entries NaN. Pass it to regression_metrics(baseline=...) -- NaNs are handled.
    """
    obs = np.asarray(observed, dtype=float)
    out = np.full_like(obs, np.nan)
    out[horizon:] = obs[:-horizon] if horizon > 0 else obs
    return out


def nse(observed, predicted):
    """Nash-Sutcliffe Efficiency.

    NOTE: this is algebraically IDENTICAL to sklearn's r2_score(observed,
    predicted) -- both are 1 - SSE/SST. Reporting both is reporting one number
    twice. If you want a genuinely different 'R2', use squared Pearson
    correlation (`pearson_r2` below), which ignores bias and scale.
    """
    o, p = np.asarray(observed, float), np.asarray(predicted, float)
    return 1.0 - np.sum((o - p) ** 2) / np.sum((o - o.mean()) ** 2)


def pearson_r2(observed, predicted):
    """Squared Pearson correlation -- the 'R2' of the hydrological literature.

    Insensitive to bias and to systematic under/over-dispersion, so it is always
    >= NSE. The GAP between pearson_r2 and NSE is exactly what bias and variance
    error cost you, and is worth reporting for that reason alone.
    """
    return float(np.corrcoef(observed, predicted)[0, 1] ** 2)


def pbias(observed, predicted):
    """Percent bias. Negative = the model under-predicts on average."""
    o, p = np.asarray(observed, float), np.asarray(predicted, float)
    return 100.0 * np.sum(p - o) / np.sum(o)


def kge(observed, predicted, decomposed=True):
    """Kling-Gupta Efficiency (Gupta et al. 2009) and its three components.

    r     : correlation           (timing / shape)
    alpha : std(pred)/std(obs)    (< 1 = over-smoothed, the classic ML failure)
    beta  : mean(pred)/mean(obs)  (bias)
    """
    o, p = np.asarray(observed, float), np.asarray(predicted, float)
    r = float(np.corrcoef(o, p)[0, 1])
    alpha = float(p.std() / o.std())
    beta = float(p.mean() / o.mean())
    k = 1.0 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)
    if decomposed:
        return {"KGE": float(k), "KGE_r": r, "KGE_alpha": alpha, "KGE_beta": beta}
    return float(k)


def peak_metrics(observed, predicted, quantile=0.95):
    """Error restricted to the highest observed values.

    Reports BIAS as well as MAE: peak MAE alone cannot tell you that the model
    systematically under-predicts events, which is the failure mode that matters
    for early warning.
    """
    o, p = np.asarray(observed, float), np.asarray(predicted, float)
    thr = ceil(np.quantile(o, quantile))
    m = o >= thr
    if m.sum() == 0:
        return {}
    q = int(round(quantile * 100))
    return {
        f"peak{q}_threshold_FTU": float(thr),
        f"peak{q}_n": int(m.sum()),
        f"peak{q}_MAE": float(mean_absolute_error(o[m], p[m])),
        f"peak{q}_bias": float(np.mean(p[m] - o[m])),      # < 0 = under-predicts peaks
        f"peak{q}_RMSE": float(np.sqrt(mean_squared_error(o[m], p[m]))),
    }


def timing_lag(observed, predicted, max_lag=12):
    """Shift at which predicted correlates best with observed.

    THE diagnostic for "is my model just doing persistence?". If the best lag
    comes out equal to the forecast horizon, the model has learned to echo the
    last observed value rather than anticipate change, and no aggregate error
    metric will reveal that.

    Returns the best lag in time steps -- POSITIVE means the predictions lag
    behind the observations (the model is reacting, not anticipating) -- and the
    correlation achieved there. A pure persistence forecast at horizon h returns
    best_lag_steps == h.
    """
    o = np.asarray(observed, float)
    p = np.asarray(predicted, float)
    o = (o - o.mean()) / (o.std() + 1e-12)
    p = (p - p.mean()) / (p.std() + 1e-12)
    best_lag, best_r = 0, -np.inf
    for lag in range(-max_lag, max_lag + 1):
        if lag > 0:
            a, b = o[lag:], p[:len(p) - lag]
        elif lag < 0:
            a, b = o[:len(o) + lag], p[-lag:]
        else:
            a, b = o, p
        if len(a) < 10:
            continue
        r = float(np.mean(a * b))
        if r > best_r:
            best_lag, best_r = lag, r
    # negate so that a positive value reads as "predictions lag observations"
    return {"best_lag_steps": int(-best_lag), "corr_at_best_lag": float(best_r)}


def exceedance_scores(observed, predicted, threshold):
    """Turn the regression into the operational question: does it cross the
    alarm level? POD / FAR / CSI are the standard warning-verification triple.
    """
    o = np.asarray(observed, float) >= threshold
    p = np.asarray(predicted, float) >= threshold
    tp = int(np.sum(o & p)); fp = int(np.sum(~o & p))
    fn = int(np.sum(o & ~p)); tn = int(np.sum(~o & ~p))
    return {
        "exc_threshold": float(threshold),
        "exc_POD": tp / (tp + fn) if tp + fn else np.nan,        # hit rate / recall
        "exc_FAR": fp / (tp + fp) if tp + fp else np.nan,        # false alarm ratio
        "exc_CSI": tp / (tp + fp + fn) if tp + fp + fn else np.nan,  # threat score
        "exc_bias": (tp + fp) / (tp + fn) if tp + fn else np.nan,    # over/under-warning
        "exc_TP": tp, "exc_FP": fp, "exc_FN": fn, "exc_TN": tn,
    }


def regression_metrics(observed, predicted, baseline=None, threshold=None,
                       peak_quantiles=(0.90, 0.95, 0.99), max_lag=12):
    """One dictionary with everything. Drops rows where any input is NaN."""
    o = np.asarray(observed, float)
    p = np.asarray(predicted, float)
    b = np.asarray(baseline, float) if baseline is not None else None

    ok = np.isfinite(o) & np.isfinite(p)
    if b is not None:
        ok &= np.isfinite(b)
    o, p = o[ok], p[ok]
    b = b[ok] if b is not None else None

    out = {
        "n": int(len(o)),
        "RMSE": float(np.sqrt(mean_squared_error(o, p))),
        "MAE": float(mean_absolute_error(o, p)),
        "MedAE": float(np.median(np.abs(o - p))),
        "NSE": float(nse(o, p)),                 # == sklearn r2_score
        "pearson_r2": pearson_r2(o, p),
        "PBIAS_pct": float(pbias(o, p)),
    }
    out.update(kge(o, p))

    # log-space error: turbidity errors scale with magnitude, so an FTU-space
    # RMSE is dominated by a handful of peaks
    if (o > -1).all() and (p > -1).all():
        out["RMSLE"] = float(np.sqrt(np.mean((np.log1p(np.clip(p, 0, None))
                                              - np.log1p(np.clip(o, 0, None))) ** 2)))

    for q in peak_quantiles:
        out.update(peak_metrics(o, p, q))

    out.update(timing_lag(o, p, max_lag=max_lag))

    # residual autocorrelation: strong lag-1 structure means the model is
    # leaving predictable dynamics on the table
    res = o - p
    if len(res) > 2:
        out["resid_acf1"] = float(np.corrcoef(res[:-1], res[1:])[0, 1])

    if threshold is not None:
        out.update(exceedance_scores(o, p, threshold))

    if b is not None:
        mse_m = float(mean_squared_error(o, p))
        mse_b = float(mean_squared_error(o, b))
        out["baseline_RMSE"] = float(np.sqrt(mse_b))
        # > 0 means you beat persistence; <= 0 means you did not
        out["skill_vs_persistence"] = float(1.0 - mse_m / mse_b) if mse_b > 0 else np.nan

    return out


# =============================================================================
# UNCERTAINTY AND MODEL COMPARISON
# =============================================================================
def block_bootstrap_ci(observed, predicted, metric_fn=None, block=24,
                       n_boot=1000, alpha=0.05, random_state=42):
    """Confidence interval for a metric on autocorrelated data.

    Resamples contiguous blocks (default 24 h) rather than individual hours, so
    the interval respects serial correlation. Without this you cannot say whether
    RF's RMSE of 8.41 is actually different from XGB's 8.52.
    """
    if metric_fn is None:
        metric_fn = lambda o, p: float(np.sqrt(mean_squared_error(o, p)))
    o = np.asarray(observed, float); p = np.asarray(predicted, float)
    ok = np.isfinite(o) & np.isfinite(p); o, p = o[ok], p[ok]

    rng = np.random.default_rng(random_state)
    n = len(o)
    n_blocks = int(np.ceil(n / block))
    starts_pool = np.arange(0, max(n - block, 1))

    vals = []
    for _ in range(n_boot):
        starts = rng.choice(starts_pool, size=n_blocks, replace=True)
        idx = np.concatenate([np.arange(s, min(s + block, n)) for s in starts])[:n]
        try:
            vals.append(metric_fn(o[idx], p[idx]))
        except Exception:
            continue
    vals = np.asarray(vals, float)
    vals = vals[np.isfinite(vals)]
    return {
        "point": float(metric_fn(o, p)),
        "lo": float(np.quantile(vals, alpha / 2)),
        "hi": float(np.quantile(vals, 1 - alpha / 2)),
        "n_boot": int(len(vals)),
    }


def diebold_mariano(observed, pred_a, pred_b, horizon=1, loss="squared"):
    """Is model A's forecast accuracy significantly different from model B's?

    The standard test in the forecasting literature. Uses a Newey-West variance
    with (horizon-1) lags and the Harvey small-sample correction.

    Negative DM statistic => A has lower loss (A is better). p < 0.05 => the
    difference is unlikely to be noise.
    """
    o = np.asarray(observed, float)
    a = np.asarray(pred_a, float)
    b = np.asarray(pred_b, float)
    ok = np.isfinite(o) & np.isfinite(a) & np.isfinite(b)
    o, a, b = o[ok], a[ok], b[ok]

    if loss == "squared":
        d = (o - a) ** 2 - (o - b) ** 2
    elif loss == "absolute":
        d = np.abs(o - a) - np.abs(o - b)
    else:
        raise ValueError("loss must be 'squared' or 'absolute'")

    n = len(d)
    dbar = d.mean()
    lag = max(horizon - 1, 0)
    gamma0 = np.sum((d - dbar) ** 2) / n
    var = gamma0
    for k in range(1, lag + 1):
        gk = np.sum((d[k:] - dbar) * (d[:-k] - dbar)) / n
        var += 2 * (1 - k / (lag + 1)) * gk
    if var <= 0:
        return {"DM": np.nan, "p_value": np.nan, "n": n,
                "better": "undetermined"}

    dm = dbar / np.sqrt(var / n)
    # Harvey, Leybourne & Newbold small-sample correction
    corr = np.sqrt((n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n)
    dm *= corr
    p = 2 * (1 - stats.t.cdf(abs(dm), df=n - 1))
    return {"DM": float(dm), "p_value": float(p), "n": int(n),
            "mean_loss_diff": float(dbar),
            "better": "A" if dbar < 0 else "B"}


# =============================================================================
# EVENT CLASSIFICATION
# =============================================================================
def classification_metrics(y_true, y_pred, y_prob=None, beta=2.0):
    """Point metrics for a rare-event classifier.

    Adds to the usual precision/recall/F1/ROC-AUC set the things that actually
    matter when positives are rare: PR-AUC (average precision), CSI, frequency
    bias, MCC, F-beta, and the base rate you are comparing against.
    """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
    fb = ((1 + beta ** 2) * precision * recall
          / (beta ** 2 * precision + recall)) if (beta ** 2 * precision + recall) else 0.0

    out = {
        "base_rate": float(y_true.mean()),
        "precision": float(precision),
        "recall_POD": float(recall),
        "F1": float(f1),
        f"F{int(beta)}": float(fb),
        "CSI_threat_score": float(tp / (tp + fp + fn)) if tp + fp + fn else np.nan,
        "FAR_false_alarm_ratio": float(fp / (tp + fp)) if tp + fp else np.nan,
        "FPR": float(fp / (fp + tn)) if fp + tn else np.nan,
        "frequency_bias": float((tp + fp) / (tp + fn)) if tp + fn else np.nan,
        "MCC": float(matthews_corrcoef(y_true, y_pred)) if len(np.unique(y_true)) > 1 else np.nan,
        "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
    }
    if y_prob is not None:
        y_prob = np.asarray(y_prob, float)
        out["ROC_AUC"] = float(roc_auc_score(y_true, y_prob))
        # PR-AUC is the honest headline for imbalanced problems; a no-skill model
        # scores base_rate here, not 0.5
        out["PR_AUC"] = float(average_precision_score(y_true, y_prob))
        out["PR_AUC_no_skill"] = float(y_true.mean())
        out["Brier"] = float(brier_score_loss(y_true, y_prob))
    return out


def threshold_sweep(y_true, y_prob, beta=2.0, n_points=101):
    """Precision / recall / F-beta / CSI across decision thresholds.

    `model.predict()` hard-codes 0.5, which is arbitrary for rare events. Pick the
    operating point on VALIDATION data (never on the test block) and report the
    curve so the precision-recall trade-off is explicit.
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, float)
    rows = []
    for t in np.linspace(0.01, 0.99, n_points):
        yp = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, yp, labels=[0, 1]).ravel()
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        fb = ((1 + beta ** 2) * pr * rc / (beta ** 2 * pr + rc)) if (beta ** 2 * pr + rc) else 0.0
        rows.append({"threshold": t, "precision": pr, "recall": rc,
                     f"F{int(beta)}": fb,
                     "CSI": tp / (tp + fp + fn) if tp + fp + fn else 0.0,
                     "TP": tp, "FP": fp, "FN": fn})
    df = pd.DataFrame(rows)
    best = df.loc[df[f"F{int(beta)}"].idxmax()]
    return df, dict(best)


def extract_events(binary_series, min_length=3):
    """Contiguous runs of 1s at least `min_length` long -> [(start, end), ...]."""
    s = np.asarray(binary_series).astype(int)
    events, start = [], None
    for i, v in enumerate(s):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            if i - start >= min_length:
                events.append((start, i - 1))
            start = None
    if start is not None and len(s) - start >= min_length:
        events.append((start, len(s) - 1))
    return events


def event_metrics(y_true, y_pred, min_event_length=3, forecast_horizon=1,
                  max_lead_search=48):
    """Event-level verification, including the half your current version omits.

    `event_based_far_with_timing` in predictor.py counts false predicted events
    but never counts MISSED observed events -- so a model that predicts almost
    nothing scores a perfect event FAR. This computes both directions, plus how
    much warning time each detected event actually got.

    Detection rule (kept from your version): an observed event of length n is
    detected if the overlapping predicted positives cover >= 1 step when n == 3,
    or >= n/2 steps when n >= 4.

    Lead time is measured from the first alarm belonging to the detecting run to
    the observed event start, then `forecast_horizon` is added, because a model
    predicting h hours ahead gives that much warning on top.
    """
    yt = np.asarray(y_true).astype(int)
    yp = np.asarray(y_pred).astype(int)

    obs_events = extract_events(yt, min_event_length)
    pred_events = extract_events(yp, min_event_length)

    def _detects(o_start, o_end):
        n = o_end - o_start + 1
        cover = int(yp[o_start:o_end + 1].sum())
        return (n == 3 and cover >= 1) or (n >= 4 and cover >= 0.5 * n)

    hits, misses, lead_times = 0, 0, []
    for o_start, o_end in obs_events:
        if _detects(o_start, o_end):
            hits += 1
            # walk back from the event start through contiguous alarms
            first = o_start
            while (first - 1 >= 0 and yp[first - 1] == 1
                   and o_start - first < max_lead_search):
                first -= 1
            if yp[o_start] == 0:               # alarm begins inside the event
                k = o_start
                while k <= o_end and yp[k] == 0:
                    k += 1
                first = k
            lead_times.append(int(o_start - first) + int(forecast_horizon))
        else:
            misses += 1

    false_events = 0
    for p_start, p_end in pred_events:
        if not any(not (p_end < o_start or p_start > o_end)
                   for o_start, o_end in obs_events):
            false_events += 1

    n_obs = len(obs_events)
    return {
        "n_observed_events": n_obs,
        "n_predicted_events": len(pred_events),
        "event_hits": hits,
        "event_misses": misses,
        "event_POD": float(hits / n_obs) if n_obs else np.nan,
        "false_events": false_events,
        "event_FAR": float(false_events / len(pred_events)) if pred_events else np.nan,
        "event_CSI": float(hits / (hits + false_events + misses))
                     if (hits + false_events + misses) else np.nan,
        "median_lead_time_h": float(np.median(lead_times)) if lead_times else np.nan,
        "mean_lead_time_h": float(np.mean(lead_times)) if lead_times else np.nan,
        "min_lead_time_h": float(np.min(lead_times)) if lead_times else np.nan,
        "lead_times_h": lead_times,
    }


# =============================================================================
# CONVENIENCE
# =============================================================================
def compare_models(observed, predictions: dict, baseline=None, threshold=None,
                   horizon=1, bootstrap=True):
    """Score several models on the same observations and test them pairwise.

    predictions: {"LSTM": arr, "RF": arr, "XGB": arr}
    Returns (metrics_table, pairwise_dm_table).
    """
    rows = []
    for name, pred in predictions.items():
        m = {"model": name}
        m.update(regression_metrics(observed, pred, baseline=baseline,
                                    threshold=threshold))
        if bootstrap:
            ci = block_bootstrap_ci(observed, pred)
            m["RMSE_lo95"], m["RMSE_hi95"] = ci["lo"], ci["hi"]
        rows.append(m)
    table = pd.DataFrame(rows).set_index("model")

    names = list(predictions)
    dm_rows = []
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            r = diebold_mariano(observed, predictions[a], predictions[b],
                                horizon=horizon)
            dm_rows.append({"A": a, "B": b, "DM": r["DM"], "p_value": r["p_value"],
                            "better": a if r["better"] == "A" else b,
                            "significant_5pct": bool(r["p_value"] < 0.05)
                                                if np.isfinite(r["p_value"]) else False})
    return table, pd.DataFrame(dm_rows)

## **Testing by reading the predictions from the training part**


In [5]:
"""
Score the saved test predictions -- no models, no scalers, no re-prediction.

Reads every  trained_models_v2/predictions/test_predictions_chrono80_w<W>_h<H>.csv
written by the training script, applies the metric functions in evaluation.py to
each model column, and writes one tidy comparison table.

Outputs (into trained_models_v2/metrics/):
  metrics_chrono80_w24_h1.csv     one file per run, one row per model
  comparison_all_runs.csv         every run stacked, with window and horizon columns

Run:  python compare_predictions.py
or:   from compare_predictions import main; results = main()
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import evaluation as ev
except ImportError:
    ev = None


# =============================================================================
# CONFIG
# =============================================================================
SAVE_ROOT = Path("trained_models_random80")
PRED_DIR = SAVE_ROOT / "predictions"
OUT_DIR = SAVE_ROOT / "metrics"

# Which prediction files to score. Narrow it to score a subset, e.g.
# "test_predictions_chrono80_w24_*.csv" for the 24 h window only.
PRED_GLOB = "test_predictions_*.csv"

# FTU columns are the ones the metrics are defined on. Set False to score the
# scaled columns instead (RMSE/MAE then have no physical meaning).
USE_FTU = True

EVENT_THRESHOLD = 23.0     # FTU, passed to regression_metrics if it takes one
BASELINE = "persistence"   # model column used as the skill reference

# Diebold-Mariano test of each model against the baseline, if evaluation.py
# provides it. Skipped silently when the signature does not match.
ADD_DM_TEST = True

# Column order in the output. Anything else the metric functions return is
# appended after these.
LEAD_COLS = ["input_window_h", "horizon_h", "split", "model", "n_obs",
             "RMSE", "MAE", "NSE", "pearson_r2", "PBIAS",
             "KGE", "KGE_r", "KGE_alpha", "KGE_beta"]


# =============================================================================
# READING THE PREDICTION FILES
# =============================================================================
#_RUN_RE = re.compile(r"test_predictions_(?P<split>.+?)_w(?P<window>\d+)_h(?P<horizon>\d+)",
#                     re.IGNORECASE)

_RUN_RE = re.compile(r"test_predictions_(?P<split>.+?)_w(?P<window>\d+)-treeL1-2-3-20-21-22-23-24_h(?P<horizon>\d+)",
                     re.IGNORECASE)

def parse_run(path):
    """'test_predictions_chrono80_w24_h1.csv' -> ('chrono80', 24, 1)."""
    m = _RUN_RE.search(Path(path).stem)
    if not m:
        raise ValueError(
            f"Cannot read window/horizon from '{Path(path).name}'. Expected "
            f"...w<window>_h<horizon>.csv")
    return m["split"], int(m["window"]), int(m["horizon"])


def read_predictions(path, use_ftu=USE_FTU):
    """Return (observed, {model: predictions}, index) from one prediction file.

    The training script writes  y_true_scaled / y_pred_<MODEL>  and an FTU copy
    of each. This picks one unit system and strips the prefixes.
    """
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    suffix = "_FTU" if use_ftu else "_scaled"

    truth_col = f"y_true{suffix}" if use_ftu else "y_true_scaled"
    if truth_col not in df.columns:
        raise KeyError(
            f"{Path(path).name}: no '{truth_col}' column. "
            f"Available: {list(df.columns)}")

    if use_ftu:
        model_cols = [c for c in df.columns
                      if c.startswith("y_pred_") and c.endswith("_FTU")]
    else:
        model_cols = [c for c in df.columns
                      if c.startswith("y_pred_") and not c.endswith("_FTU")]
    if not model_cols:
        raise KeyError(f"{Path(path).name}: no model columns found.")

    keep = [truth_col] + model_cols
    clean = df[keep].replace([np.inf, -np.inf], np.nan).dropna()
    dropped = len(df) - len(clean)
    if dropped:
        print(f"    dropped {dropped:,} rows with missing or infinite values")

    def model_name(col):
        name = col[len("y_pred_"):]
        return name[:-len("_FTU")] if name.endswith("_FTU") else name

    preds = {model_name(c): clean[c].to_numpy(dtype=float) for c in model_cols}
    return clean[truth_col].to_numpy(dtype=float), preds, clean.index


# =============================================================================
# METRICS
# =============================================================================
def _metric(fname):
    """Fetch a metric function from evaluation.py or the calling namespace."""
    fn = getattr(ev, fname, None) if ev is not None else None
    return fn if fn is not None else globals().get(fname)


def score(observed, predicted, baseline=None):
    """Metrics for one model. Uses evaluation.regression_metrics when present,
    otherwise assembles the same numbers from the individual functions."""
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    full = _metric("regression_metrics")
    if full is not None:
        try:
            return dict(full(observed, predicted, baseline=baseline,
                             threshold=EVENT_THRESHOLD))
        except TypeError:
            return dict(full(observed, predicted))

    from sklearn.metrics import mean_absolute_error, mean_squared_error
    rmse = float(np.sqrt(mean_squared_error(observed, predicted)))
    out = {
        "RMSE": rmse,
        "MAE": float(mean_absolute_error(observed, predicted)),
        "NSE": float(_metric("nse")(observed, predicted)),
        "pearson_r2": float(_metric("pearson_r2")(observed, predicted)),
        "PBIAS": float(_metric("pbias")(observed, predicted)),
    }
    out.update(_metric("kge")(observed, predicted, decomposed=True))

    if baseline is not None:
        base = np.asarray(baseline, dtype=float)
        ok = ~np.isnan(base)
        base_rmse = float(np.sqrt(mean_squared_error(observed[ok], base[ok])))
        out["RMSE_baseline"] = base_rmse
        out["skill_vs_baseline"] = (1.0 - rmse / base_rmse
                                    if base_rmse else np.nan)
    return out


def dm_vs_baseline(observed, predicted, baseline, horizon):
    """Diebold-Mariano against the baseline, if evaluation.py provides it.
    Returns {} when the function is absent or the signature does not match."""
    dm = _metric("diebold_mariano")
    if dm is None or baseline is None:
        return {}
    for kwargs in ({"horizon": horizon}, {"h": horizon}, {}):
        try:
            res = dm(observed, predicted, baseline, **kwargs)
        except Exception:
            continue
        if isinstance(res, dict):
            return {f"DM_{k}": v for k, v in res.items()}
        if isinstance(res, (tuple, list)) and len(res) >= 2:
            return {"DM_stat": float(res[0]), "DM_pvalue": float(res[1])}
        try:
            return {"DM_stat": float(res)}
        except (TypeError, ValueError):
            return {}
    return {}


# =============================================================================
# MAIN
# =============================================================================
def score_file(path):
    """One prediction file -> a list of metric rows, one per model."""
    split, window, horizon = parse_run(path)
    observed, preds, index = read_predictions(path)

    print(f"\n{Path(path).name}")
    print(f"  window {window} h, horizon {horizon} h, {len(observed):,} rows "
          f"({index.min()} -> {index.max()})")
    print(f"  observed range: {observed.min():.2f} to {observed.max():.2f}"
          f"{' FTU' if USE_FTU else ' (scaled)'}")

    base = preds.get(BASELINE)
    if base is None:
        print(f"  note: no '{BASELINE}' column -- skill scores omitted")

    rows = []
    for name, yhat in preds.items():
        row = {"input_window_h": window, "horizon_h": horizon, "split": split,
               "model": name, "n_obs": len(observed)}
        row.update(score(observed, yhat, baseline=base))
        if ADD_DM_TEST and name != BASELINE:
            row.update(dm_vs_baseline(observed, yhat, base, horizon))
        rows.append(row)
        print(f"    {name:12s} RMSE={row.get('RMSE', float('nan')):8.3f}  "
              f"NSE={row.get('NSE', float('nan')):7.4f}  "
              f"KGE={row.get('KGE', float('nan')):7.4f}")
    return rows


def order_columns(df):
    lead = [c for c in LEAD_COLS if c in df.columns]
    return df[lead + [c for c in df.columns if c not in lead]]


def main():
    files = sorted(PRED_DIR.glob(PRED_GLOB))
    if not files:
        raise SystemExit(
            f"No prediction files matching '{PRED_GLOB}' in {PRED_DIR.resolve()}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Scoring {len(files)} prediction file(s) from {PRED_DIR}")

    all_rows = []
    for path in files:
        try:
            rows = score_file(path)
        except (ValueError, KeyError) as exc:
            print(f"\n{Path(path).name}: SKIPPED -- {exc}")
            continue
        all_rows += rows

        # per-run table, named for its window and horizon
        split, window, horizon = parse_run(path)
        run_df = order_columns(pd.DataFrame(rows)).sort_values("RMSE")
        run_csv = OUT_DIR / f"metrics_{split}_w{window}_h{horizon}.csv"
        run_df.to_csv(run_csv, index=False, float_format="%.6f")
        print(f"  -> {run_csv}")

    if not all_rows:
        raise SystemExit("Nothing was scored.")

    results = order_columns(pd.DataFrame(all_rows))
    results = results.sort_values(["input_window_h", "horizon_h", "RMSE"])
    combined = OUT_DIR / "comparison_all_runs_reduced.csv"
    results.to_csv(combined, index=False, float_format="%.6f")

    print("\n" + "=" * 78)
    print(f"SUMMARY -- {'FTU' if USE_FTU else 'scaled'} units, "
          f"skill relative to '{BASELINE}'")
    print("=" * 78)
    print(results.round(4).to_string(index=False))
    print(f"\nPer-run tables -> {OUT_DIR}")
    print(f"Combined table -> {combined}")
    return results


if __name__ == "__main__":
    main()

Scoring 21 prediction file(s) from trained_models_random80/predictions

test_predictions_random80_w24-treeL1-2-3-20-21-22-23-24_h6_f2.csv
  window 24 h, horizon 6 h, 6,938 rows (2020-01-02 09:00:00 -> 2024-02-26 19:00:00)
  observed range: 3.81 to 106.99 FTU
    persistence  RMSE=   5.805  NSE= 0.4025  KGE= 0.7030
    RF           RMSE=   4.290  NSE= 0.6737  KGE= 0.7057
    XGB          RMSE=   4.445  NSE= 0.6497  KGE= 0.6920
  -> trained_models_random80/metrics/metrics_random80_w24_h6.csv

test_predictions_random80_w24-treeL1-2-3-20-21-22-23-24_h6_f4.csv
  window 24 h, horizon 6 h, 6,938 rows (2020-01-02 09:00:00 -> 2024-02-26 19:00:00)
  observed range: 3.81 to 106.99 FTU
    persistence  RMSE=   5.805  NSE= 0.4025  KGE= 0.7030
    RF           RMSE=   4.066  NSE= 0.7068  KGE= 0.7229
  -> trained_models_random80/metrics/metrics_random80_w24_h6.csv

test_predictions_random80_w24-treeL1-2-3-20-21-22-23-24_h6_f5.csv
  window 24 h, horizon 6 h, 6,938 rows (2020-01-02 09:00:00 -> 2024-02-

## **Testing by running the trained models**


In [ ]:
# -*- coding: utf-8 -*-
"""
Run trained turbidity models on a test set and score them all identically.

Outputs two CSVs per run:
  predictions_<tag>.csv   one row per test timestamp: observed, persistence,
                          and one column per model
  metrics_<tag>.csv       one row per model, one column per metric

Edit the CONFIG block, then:  python evaluate_models.py
In a notebook: import the module and call main(), or paste the blocks in order.
"""

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# The metric functions. Works either way: as a module next to this file, or
# with the functions already defined in the notebook namespace.
try:
    import evaluation as ev
except ImportError:
    ev = None


# =============================================================================
# CONFIG -- edit this block only
# =============================================================================
TEST_CSV = Path("data/test_LSTMRF.csv")

TARGET = "sensor_turbidity"
FEATURE_COLS = [
    "sensor_turbidity",
     "Hag_dis",
     "sensor_temperature",
     "Nieu_EC",
     "Lob_turbidity",
     "flow_direction",
     "month",
]

INPUT_WINDOW = 24        # config['seq_length (hours)']
FORECAST_HORIZON = 1     # config['forecasting horizon (hours)']
EVENT_THRESHOLD = 25.0   # FTU, for the event-based metrics

SPLIT_TAG = "chrono80"
RUN_TAG = f"{len(FEATURE_COLS)}feat_w{INPUT_WINDOW}_h{FORECAST_HORIZON}"

# A DIRECTORY. The two CSV filenames are built from RUN_TAG below.
OUT_DIR = Path("trained_models_v2/results") / RUN_TAG

_ROOT = Path("trained_models_v2")
_STEM = f"{SPLIT_TAG}_w{INPUT_WINDOW}_h{FORECAST_HORIZON}"

# name -> path. Comment out anything you are not scoring this run.
# Run the "find the model files" snippet below if any of these are skipped.
MODEL_PATHS = {
    "RF":   _ROOT / f"RF models/{_STEM}/rf_{_STEM}.joblib",
    "XGB":  _ROOT / f"XGB models/{_STEM}/xgb_{_STEM}.joblib",
    "LSTM": _ROOT / f"LSTM models/{_STEM}/lstm_{_STEM}.pth",
}

# The .pth files hold PyTorch weights, not a pickled model object. If the saved
# architecture is the usual nn.LSTM + one Linear head, the script rebuilds it
# from the weight shapes automatically. If it is anything else, import your own
# class and set this to a zero-argument factory, e.g.
#   from my_models import TurbidityLSTM
#   LSTM_FACTORY = lambda: TurbidityLSTM(input_size=len(FEATURE_COLS), hidden_size=64)
LSTM_FACTORY = None

# Only models that take a 3-D sequence (samples, timesteps, features).
# RF and XGB take the flat lag matrix and will crash if listed here.
SEQUENCE_MODELS = {"LSTM"}

# The scaler used for all three models. May hold a single fitted scaler or a
# dict of per-variable scalers ({"sensor_turbidity": MinMaxScaler(), ...});
# both are handled.
SCALER_PATH = _ROOT / f"scaler_dict_{SPLIT_TAG}.pkl"

# TEST_CSV is already normalised, so the features go to the models untouched.
# Set False only if you switch to a file in raw FTU, in which case the script
# scales the lag matrix before predicting.
TEST_DATA_IS_SCALED = True

# Models trained on a scaled target predict in scaled units; their output is
# inverse-transformed before scoring. All three were trained that way.
MODELS_PREDICT_SCALED = {"RF", "XGB", "LSTM"}


# =============================================================================
# FIND THE MODEL FILES  (run this once if models are being skipped)
# =============================================================================
def list_model_files(root=_ROOT):
    """Print every saved model/scaler under `root` so MODEL_PATHS can be fixed."""
    root = Path(root)
    if not root.exists():
        print(f"{root.resolve()} does not exist -- are you in the right cwd?")
        return []
    found = sorted(p for p in root.rglob("*")
                   if p.is_file() and p.suffix in {".pkl", ".joblib", ".h5",
                                                   ".keras", ".pt", ".pth"})
    for p in found:
        print(p)
    if not found:
        print(f"No model files under {root.resolve()}")
    return found


# =============================================================================
# FEATURE BUILDING
# =============================================================================
def create_ml_data(data, input_window, forecast_horizon, base_feature_columns,
                   target=TARGET):
    """Lagged design matrix and forward-shifted target, NaN rows dropped.

    Returns (X_df, y_series, var_names, n_lags). X_df keeps the DatetimeIndex so
    predictions stay tied to real timestamps. Column order is var-major:
    var0_lag{k} ... var0_lag{W}, var1_lag{k} ..., which reshape_to_sequences
    below relies on.
    """
    missing = [v for v in base_feature_columns if v not in data.columns]
    if missing:
        raise KeyError(f"Features missing from the test file: {missing}")

    start_lag = forecast_horizon if forecast_horizon > 0 else 1
    lags = list(range(start_lag, input_window + 1))
    if not lags:
        raise ValueError(
            f"forecast_horizon ({forecast_horizon}) exceeds input_window "
            f"({input_window}), so there are no usable lags.")

    lag_cols = {
        f"{var}_lag{lag}": data[var].shift(lag)
        for var in base_feature_columns
        for lag in lags
    }
    X_df = pd.DataFrame(lag_cols, index=data.index)
    y_series = data[target].shift(-forecast_horizon).rename("observed")

    combined = pd.concat([X_df, y_series], axis=1).dropna()
    return (combined[X_df.columns], combined["observed"],
            list(base_feature_columns), len(lags))


def reshape_to_sequences(X_df, var_names, n_lags):
    """Flat lag matrix -> (samples, timesteps, features), oldest timestep first.

    CHECK THIS AGAINST YOUR TRAINING CODE. If the LSTM was trained with the
    feature axis and time axis swapped, drop the final transpose.
    """
    n = len(X_df)
    arr = np.asarray(X_df, dtype=float).reshape(n, len(var_names), n_lags)
    arr = arr[:, :, ::-1]                 # lag W .. lag k  ->  oldest .. newest
    # ascontiguousarray: the reversed slice has negative strides, which torch
    # refuses to wrap in a tensor.
    return np.ascontiguousarray(np.transpose(arr, (0, 2, 1)))


# =============================================================================
# SCALING
# =============================================================================
def _var_of(col):
    """'Hag_dis_lag7' -> 'Hag_dis'"""
    return col.rsplit("_lag", 1)[0]


def scale_features(X_df, scaler):
    """Apply a single scaler, or a dict of per-variable scalers, to the lag
    matrix. Returns a DataFrame with the same index and columns."""
    if scaler is None:
        return X_df
    if hasattr(scaler, "transform"):
        return pd.DataFrame(scaler.transform(X_df.to_numpy(dtype=float)),
                            index=X_df.index, columns=X_df.columns)
    if isinstance(scaler, dict):
        out = X_df.copy()
        for col in X_df.columns:
            var = _var_of(col)
            s = scaler.get(var)
            if s is None:
                raise KeyError(
                    f"No scaler for '{var}' in the scaler dict "
                    f"(has: {sorted(scaler)[:8]}...)")
            out[col] = s.transform(X_df[[col]].to_numpy(dtype=float)).ravel()
        return out
    raise TypeError(f"Cannot use {type(scaler).__name__} as an X scaler.")


def inverse_target(yhat, scaler, target=TARGET):
    """Undo target scaling so the metrics see FTU."""
    if scaler is None:
        return yhat
    s = scaler if hasattr(scaler, "inverse_transform") else None
    if s is None and isinstance(scaler, dict):
        s = scaler.get(target)
        if s is None:
            raise KeyError(f"No scaler for target '{target}' in the scaler dict.")
    return s.inverse_transform(np.asarray(yhat).reshape(-1, 1)).ravel()


# =============================================================================
# MODEL LOADING AND PREDICTION
# =============================================================================
_MAGIC = [
    (b"\x80",                   "pickle protocol 2+ -- joblib.load should work"),
    (b"\x1f\x8b",               "gzip-compressed pickle -- joblib.load handles it"),
    (b"ZF",                     "joblib zlib-compressed -- joblib.load handles it"),
    (b"x\x9c",                  "zlib-compressed pickle -- joblib.load handles it"),
    (b"x^",                     "zlib-compressed pickle -- joblib.load handles it"),
    (b"x\x01",                  "zlib-compressed pickle -- joblib.load handles it"),
    (b"x\xda",                  "zlib-compressed pickle -- joblib.load handles it"),
    (b"BZh",                    "bz2-compressed pickle -- joblib.load handles it"),
    (b"\xfd7zXZ",               "xz-compressed pickle -- joblib.load handles it"),
    (b"\x04\x22\x4d\x18",       "lz4-compressed pickle -- joblib.load handles it"),
    (b"PK\x03\x04",             "ZIP -- torch.save(>=1.6) or a .keras file, NOT a pickle. "
                                "Load with torch.load / keras.models.load_model."),
    (b"\x89HDF\r\n\x1a\n",      "HDF5 -- a Keras .h5 model. "
                                "Load with keras.models.load_model."),
    (b"version https://git-lfs", "GIT LFS POINTER -- the real file was never "
                                "downloaded. Run `git lfs pull`."),
    (b"<!DOCTYPE",              "HTML -- an error page was saved instead of the model."),
    (b"<html",                  "HTML -- an error page was saved instead of the model."),
]


def inspect_file(path):
    """Say what a file actually is, based on its first bytes."""
    path = Path(path)
    if not path.exists():
        return f"{path}: MISSING"
    head = path.open("rb").read(64)
    size = path.stat().st_size
    for magic, what in _MAGIC:
        if head.startswith(magic):
            return f"{path.name} ({size:,} B): {what}"
    if head[:1] in (b"(", b"}", b"]", b"c", b"d"):
        return (f"{path.name} ({size:,} B): pickle protocol 0/1 (text-mode) -- "
                f"valid, but load with pickle.load, not joblib")
    try:
        preview = head.decode("ascii").splitlines()[0][:50]
        return (f"{path.name} ({size:,} B): plain TEXT, starts {preview!r} -- "
                f"not a pickle at all")
    except UnicodeDecodeError:
        return (f"{path.name} ({size:,} B): unrecognised binary, "
                f"first bytes {head[:8]!r}")


def diagnose_files():
    """Report what every configured model/scaler file really contains."""
    for name, path in MODEL_PATHS.items():
        print(f"  {name:5s} {inspect_file(path)}")
    print(f"  {'SCALER':5s} {inspect_file(SCALER_PATH)}")


def describe_state_dict(path):
    """Print the parameter names and shapes inside a .pth file."""
    import torch
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(obj, dict):
        print(f"{Path(path).name}: a full {type(obj).__name__} object, not a state dict")
        return obj
    state = obj.get("model_state_dict", obj.get("state_dict", obj))
    for k, v in state.items():
        print(f"  {k:40s} {tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__}")
    extra = [k for k in obj if k not in ("model_state_dict", "state_dict")]
    if extra and state is not obj:
        print(f"  other checkpoint keys: {extra}")
    return state


def _build_lstm_from_state(state):
    """Rebuild the common 'nn.LSTM + one Linear head' architecture from weight
    shapes, so the .pth loads without needing the original class."""
    import torch
    from torch import nn

    ih0 = [k for k in state if k.endswith("weight_ih_l0")]
    if len(ih0) != 1:
        raise RuntimeError(
            f"Expected exactly one LSTM stack, found {len(ih0)}: {ih0}. "
            f"Set LSTM_FACTORY to your own model class.")
    lstm_prefix = ih0[0][: -len("weight_ih_l0")]
    w = state[ih0[0]]
    input_size, hidden = w.shape[1], w.shape[0] // 4
    bidir = any("_reverse" in k for k in state)
    layers = 1 + max(
        int(k.rsplit("_l", 1)[1].replace("_reverse", ""))
        for k in state if "weight_ih_l" in k)

    head = [k for k in state
            if k.endswith("weight") and not k.startswith(lstm_prefix)
            and state[k].ndim == 2]
    if len(head) != 1:
        raise RuntimeError(
            f"Expected one Linear head, found {head}. "
            f"Set LSTM_FACTORY to your own model class.")
    head_prefix = head[0][: -len("weight")]
    out_features = state[head[0]].shape[0]

    class RebuiltLSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hidden, num_layers=layers,
                                batch_first=True, bidirectional=bidir)
            self.head = nn.Linear(hidden * (2 if bidir else 1), out_features)

        def forward(self, x):
            out, _ = self.lstm(x)
            return self.head(out[:, -1, :])   # last timestep

    remapped = {}
    for k, v in state.items():
        if k.startswith(lstm_prefix):
            remapped["lstm." + k[len(lstm_prefix):]] = v
        elif k.startswith(head_prefix):
            remapped["head." + k[len(head_prefix):]] = v
        else:
            raise RuntimeError(
                f"Unexpected parameter '{k}' -- the architecture is not a plain "
                f"LSTM + Linear. Set LSTM_FACTORY to your own model class.")

    model = RebuiltLSTM()
    model.load_state_dict(remapped)
    model.eval()
    print(f"    rebuilt LSTM: input_size={input_size}, hidden={hidden}, "
          f"layers={layers}, bidirectional={bidir}")
    return model


def _load_torch(path):
    import torch
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if hasattr(obj, "state_dict") and hasattr(obj, "eval"):   # a full module
        obj.eval()
        return obj
    state = obj.get("model_state_dict", obj.get("state_dict", obj))
    if LSTM_FACTORY is not None:
        model = LSTM_FACTORY()
        model.load_state_dict(state)
        model.eval()
        return model
    return _build_lstm_from_state(state)


def load_model(path):
    """joblib first, then torch, then keras -- whatever the file turns out to be."""
    if not path.exists():
        raise FileNotFoundError(path)
    head = path.open("rb").read(8)
    if path.suffix in (".pth", ".pt"):
        return _load_torch(path)
    try:
        return joblib.load(path)
    except Exception as exc:
        if head.startswith(b"PK\x03\x04"):          # torch zip or .keras
            try:
                return _load_torch(path)
            except Exception:
                import keras
                return keras.models.load_model(path)
        if head.startswith(b"\x89HDF"):             # Keras .h5
            import keras
            return keras.models.load_model(path)
        raise RuntimeError(f"{inspect_file(path)}\n  original error: {exc}") from exc


def predict(model, X):
    """Predict with an sklearn, xgboost, keras or torch model."""
    if hasattr(model, "eval") and hasattr(model, "state_dict"):        # torch
        import torch
        model.eval()
        with torch.no_grad():
            out = model(torch.as_tensor(np.asarray(X), dtype=torch.float32))
        return np.asarray(out.cpu().numpy()).ravel()
    return np.asarray(model.predict(X)).ravel()                        # the rest


def align_tabular(name, model, X_df):
    """Put the lag columns in the order the model was trained on, and fail
    early if the widths disagree. Models fitted on plain arrays get an array
    back, which also avoids sklearn's 'X has feature names' warning."""
    names = getattr(model, "feature_names_in_", None)
    if names is not None:
        missing = [c for c in names if c not in X_df.columns]
        if missing:
            raise KeyError(f"{name}: lag columns missing from test set: {missing}")
        return X_df[list(names)]

    expected = getattr(model, "n_features_in_", None)
    if expected is None:
        booster = getattr(model, "get_booster", None)
        if booster is not None:
            expected = getattr(booster(), "num_features", lambda: None)()
    if expected is not None and X_df.shape[1] != expected:
        raise ValueError(
            f"{name}: test matrix has {X_df.shape[1]} features but the model "
            f"was trained on {expected}. INPUT_WINDOW, FORECAST_HORIZON or "
            f"FEATURE_COLS does not match training.")
    return X_df.to_numpy(dtype=float)


# =============================================================================
# METRICS
# =============================================================================
def _metric(fname):
    """Fetch a metric function from evaluation.py or the notebook namespace."""
    fn = getattr(ev, fname, None) if ev is not None else None
    return fn if fn is not None else globals().get(fname)


def score(observed, predicted, baseline):
    """One metric dict per model. Uses regression_metrics when available;
    otherwise assembles the same numbers from the individual functions."""
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    full = _metric("regression_metrics")
    if full is not None:
        return dict(full(observed, predicted, baseline=baseline,
                         threshold=EVENT_THRESHOLD))

    from sklearn.metrics import mean_absolute_error, mean_squared_error
    rmse = float(np.sqrt(mean_squared_error(observed, predicted)))
    out = {
        "RMSE": rmse,
        "MAE": float(mean_absolute_error(observed, predicted)),
        "NSE": float(_metric("nse")(observed, predicted)),
        "pearson_r2": float(_metric("pearson_r2")(observed, predicted)),
        "PBIAS": float(_metric("pbias")(observed, predicted)),
    }
    out.update(_metric("kge")(observed, predicted, decomposed=True))

    baseline = np.asarray(baseline, dtype=float)
    ok = ~np.isnan(baseline)
    base_rmse = float(np.sqrt(mean_squared_error(observed[ok], baseline[ok])))
    out["RMSE_persistence"] = base_rmse
    out["skill_vs_persistence"] = 1.0 - rmse / base_rmse if base_rmse else np.nan
    return out


# =============================================================================
# MAIN
# =============================================================================
def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    raw = pd.read_csv(TEST_CSV, index_col=0, parse_dates=True).sort_index()
    X_df, y, var_names, n_lags = create_ml_data(
        raw, INPUT_WINDOW, FORECAST_HORIZON, FEATURE_COLS)
    observed = y.to_numpy(dtype=float)

    print(f"{len(X_df)} test rows, {X_df.shape[1]} lag features "
          f"({len(var_names)} vars x {n_lags} lags), horizon {FORECAST_HORIZON}h")

    scaler = load_model(SCALER_PATH) if SCALER_PATH else None
    if scaler is None and (TEST_DATA_IS_SCALED or MODELS_PREDICT_SCALED):
        raise SystemExit(f"Scaled data/models need a scaler at {SCALER_PATH}")

    # The test file is normalised, so the observed series is too. Metrics are
    # defined on FTU, so bring the truth back to FTU once, here.
    if TEST_DATA_IS_SCALED:
        observed = inverse_target(observed, scaler)
    print(f"observed range: {observed.min():.2f} to {observed.max():.2f} FTU "
          f"(sanity check -- turbidity units, not 0-1)")

    # Features go in as they are when the file is already normalised.
    X_model = X_df if TEST_DATA_IS_SCALED else scale_features(X_df, scaler)

    preds = pd.DataFrame({"observed": observed}, index=y.index)
    preds["persistence"] = _metric("persistence_forecast")(
        observed, horizon=FORECAST_HORIZON)

    rows, skipped = {}, []
    for name, path in MODEL_PATHS.items():
        try:
            model = load_model(path)
        except FileNotFoundError:
            print(f"  {name}: SKIPPED, no file at {path}")
            skipped.append(name)
            continue

        X = (reshape_to_sequences(X_model, var_names, n_lags)
             if name in SEQUENCE_MODELS
             else align_tabular(name, model, X_model))

        yhat = predict(model, X)
        if name in MODELS_PREDICT_SCALED:
            yhat = inverse_target(yhat, scaler)

        if len(yhat) != len(observed):
            print(f"  {name}: SKIPPED, returned {len(yhat)} predictions for "
                  f"{len(observed)} rows")
            skipped.append(name)
            continue

        preds[name] = yhat
        rows[name] = score(observed, yhat, preds["persistence"].to_numpy())
        print(f"  {name}: RMSE {rows[name].get('RMSE', float('nan')):.3f}")

    if not rows:
        print("\nNo models were scored. The saved files are:")
        list_model_files()
        raise SystemExit("Fix MODEL_PATHS to match the paths above.")

    ok = preds["persistence"].notna().to_numpy()
    base = preds["persistence"].to_numpy()
    rows["persistence"] = score(observed[ok], base[ok], base[ok])

    metrics = pd.DataFrame(rows).T.rename_axis("model")

    pred_path = OUT_DIR / f"predictions_{RUN_TAG}.csv"
    met_path = OUT_DIR / f"metrics_{RUN_TAG}.csv"
    preds.to_csv(pred_path, index_label="timestamp")
    metrics.to_csv(met_path, float_format="%.6f")

    if skipped:
        print(f"\nNot scored: {', '.join(skipped)}")
    print(f"\nWrote {pred_path} and {met_path}\n")
    print(metrics.round(4).to_string())
    return metrics


if __name__ == "__main__":
    main()

KeyError: "Features missing from the test file: ['month']"